# Model create and save

In [1]:
# After clean data
import pandas as pd
df =pd.read_csv("House_Cleaned_data.csv")

In [2]:
# train test split

from sklearn.model_selection import train_test_split

x = df.drop(columns=["price"])
y = df["price"]

X_train, X_test,Y_train,Y_test = train_test_split(x,y,test_size=0.2,random_state= 42)

# 2nd way
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
# task 1 transform applying
# columns transform
columns_trans = ColumnTransformer(
    [('onehot_location', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['location']),
     ('onehot_area_type', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ["area_type"]),
     ('onehot_availability', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ["availability"]),
     
     ('scaler', StandardScaler(), ["total_sqft", "bath"]),

     ],
    remainder='passthrough')

# task 2 model apply
# model
lr = LinearRegression()
#pipeline
pipe = make_pipeline(columns_trans,lr)
print(pipe)

pipe.fit(X_train,Y_train)
pipe.score(X_train,Y_train)
# trainig accuracy
# Predictions
y_pred = pipe.predict(X_test)

# Performance Matrix
# Measuring Performance metrics-Lost and Cost Function (MAE,MSE,RMSE,R2 Score)
# cost functions --> calculate erros
from sklearn.metrics import mean_absolute_error, mean_squared_error,root_mean_squared_error

print("MAE:",mean_absolute_error(Y_test,y_pred))
print("MSE:",mean_squared_error(Y_test,y_pred))
print("RMSE:",root_mean_squared_error(Y_test,y_pred))

# R2 score
from sklearn.metrics import r2_score
r2_score(Y_test,y_pred)
# save mode
import pickle
pickle.dump(pipe,open("model_Final.pkl","wb"))

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehot_location',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['location']),
                                                 ('onehot_area_type',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['area_type']),
                                                 ('onehot_availability',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                  

In [3]:
d = x.sample(1)
d

,area_type,availability,location,size_BHK,total_sqft,bath,balcony
1529,Built-up Area,Ready To Move,Chikkasandra,2,1070.0,2.0,2.0


## Backend developer

In [4]:
# Load the model

import pickle
model = pickle.load(open("model_final.pkl","rb"))

area_type = input("enter your area type: ")
availability = input("enter your availability time : ")
location = input("enter your location : ")
size_bhk = int(input("enter the BHK size(number of bedroom): "))
total_sqft = float(input("enter the total size in sqft :"))
bath = int(input("enter the bathroom (number of bathroom): "))
balcony = int(input("enter the balcony (number of balcony): "))

d = [[area_type,availability,location,size_bhk,total_sqft,bath,balcony]]
d

[['Built-up  Area', 'Ready To Move', 'Chikkasandra', 2, 1070.0, 2, 2]]

In [5]:
x.columns

Index(['area_type', 'availability', 'location', 'size_BHK', 'total_sqft',
       'bath', 'balcony'],
      dtype='str')

In [6]:
d = pd.DataFrame(d,columns=x.columns)
ans = model.predict(d)
print(f"The House price based on your data is {round(ans[0],2)} ₹Lakh only")

The House price based on your data is -49.87 ₹Lakh only


# Gradio


In [ ]:
%pip install gradio

In [9]:
import pickle
import pandas as pd
import gradio as gr
# =========================
# Load Model
# =========================
with open("model_Final.pkl", "rb") as file:
    model = pickle.load(file)

# =========================
# Prediction Function
# =========================
def predict_house_price(area_type,availability,location,size_bhk,total_sqft,bath,balcony):
    try:
        # Create input data
        data = [[area_type,availability,location,int(size_bhk),float(total_sqft),int(bath),int(balcony)]]

        # IMPORTANT:
        # These column names must be exactly the same
        # as the columns used during model training.
        columns = ["area_type","availability","location","size_bhk","total_sqft","bath","balcony"]

        df = pd.DataFrame(data, columns=columns)

        # Prediction
        prediction = model.predict(df)

        price = round(prediction[0], 2)

        return f"🏠 Estimated House Price: ₹{price} Lakh"

    except Exception as e:
        return f"❌ Error: {str(e)}"
# =========================
# Gradio Interface
# =========================
with gr.Blocks(title="House Price Prediction") as app:

    gr.Markdown(
        """
        # 🏠 House Price Prediction

        Enter the property details below to predict the
        estimated house price.
        """
    )
    with gr.Row():

        with gr.Column():
            area_type = gr.Dropdown(choices=["Super built-up Area","Built-up Area","Plot Area","Carpet Area"],
                        label="Area Type",value="Super built-up Area")

            availability = gr.Textbox(label="Availability",placeholder="Example: Ready To Move")
            location = gr.Textbox(label="Location",placeholder="Example: Whitefield")
            size_bhk = gr.Number(label="BHK Size",minimum=1,value=2)

        with gr.Column():

            total_sqft = gr.Number(label="Total Size (sqft)",minimum=100,value=1000)
            bath = gr.Number(label="Number of Bathrooms",minimum=1,value=2)
            balcony = gr.Number(label="Number of Balconies",minimum=0,value=1)

    predict_button = gr.Button("Predict House Price",variant="primary")

    output = gr.Textbox(label="Prediction",interactive=False)

    predict_button.click(
        fn=predict_house_price,
        inputs=[area_type,availability,location,size_bhk,total_sqft,bath,balcony],
        outputs=output)
# =========================
# Launch App
# =========================

if __name__ == "__main__":
    app.launch()

c:\Users\lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
locations = df["location"].unique()
locations
# save the data for future use
pickle.dump(locations, open("locations.pkl","wb"))

# carpet area
carpet_area = df["area_type"].unique()
carpet_area
# save the data for future use
pickle.dump(carpet_area, open("area_type.pkl","wb"))

# carpet area
available = df["availability"].unique()
available
# save the data for future use
pickle.dump(available, open("avalibility.pkl","wb"))

In [11]:
locations = pickle.load(open("locations.pkl", "rb"))
area_types = pickle.load(open("area_type.pkl", "rb"))
available = pickle.load (open("avalibility.pkl","rb"))

In [12]:
import pickle
import pandas as pd
import gradio as gr
# =========================
# Load Model
# =========================
with open("model_Final.pkl", "rb") as file:
    model = pickle.load(file)

with open("locations.pkl", "rb") as file:
    locations = pickle.load(file)
    locations = list(locations)

with open("area_type.pkl", "rb") as file:
    area_types = pickle.load(file)
    area_types= list(area_types)


with open("avalibility.pkl", "rb") as file:
    available = pickle.load(file)
    available =list(available)

# =========================
# Prediction Function
# =========================
def predict_house_price(area_type,availability,location,size_bhk,total_sqft,bath,balcony):
    try:
        # Create input data
        data = [[area_type,availability,location,int(size_bhk),float(total_sqft),int(bath),int(balcony)]]

        # IMPORTANT:
        # These column names must be exactly the same
        # as the columns used during model training.
        columns = ["area_type","availability","location","size_bhk","total_sqft","bath","balcony"]

        df = pd.DataFrame(data, columns=columns)

        # Prediction
        prediction = model.predict(df)

        price = round(prediction[0], 2)

        return f"🏠 Estimated House Price: ₹{price} Lakh"

    except Exception as e:
        return f"❌ Error: {str(e)}"
# =========================
# Gradio Interface
# =========================
with gr.Blocks(title="House Price Prediction") as app:

    gr.Markdown(
        """
        # 🏠 House Price Prediction

        Enter the property details below to predict the
        estimated house price.
        """
    )
    with gr.Row():

        with gr.Column():
            area_type = gr.Dropdown(choices=area_types,
                        label="Area Type",value="Super built-up Area")

            availability = gr.Dropdown(choices=available, label="Availability")
            location = gr.Dropdown (choices= locations, label="Location")
            size_bhk = gr.Number(label="BHK Size",minimum=1,value=2)

        with gr.Column():

            total_sqft = gr.Number(label="Total Size (sqft)",minimum=100,value=1000)
            bath = gr.Number(label="Number of Bathrooms",minimum=1,value=2)
            balcony = gr.Number(label="Number of Balconies",minimum=0,value=1)

    predict_button = gr.Button("Predict House Price",variant="primary")

    output = gr.Textbox(label="Prediction",interactive=False)

    predict_button.click(
        fn=predict_house_price,
        inputs=[area_type,availability,location,size_bhk,total_sqft,bath,balcony],
        outputs=output)
# =========================
# Launch App
# =========================

if __name__ == "__main__":
    app.launch()

c:\Users\lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\components\dropdown.py:235: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Super built-up Area or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [13]:
import pickle
import pandas as pd
import gradio as gr

# =========================
# Load Model
# =========================

with open("model_Final.pkl", "rb") as file:
    model = pickle.load(file)


# =========================
# Load Dropdown Values
# =========================

with open("locations.pkl", "rb") as file:
    locations = pickle.load(file)

with open("area_type.pkl", "rb") as file:
    area_types = pickle.load(file)

with open("avalibility.pkl", "rb") as file:
    available = pickle.load(file)


# =========================
# Prediction Function
# =========================

def predict_house_price(
    area_type,
    availability,
    location,
    size_bhk,
    total_sqft,
    bath,
    balcony
):

    data = [[
        area_type,
        availability,
        location,
        int(size_bhk),
        float(total_sqft),
        int(bath),
        int(balcony)
    ]]

    columns = [
        "area_type",
        "availability",
        "location",
        "size_bhk",
        "total_sqft",
        "bath",
        "balcony"
    ]

    df = pd.DataFrame(data, columns=columns)

    prediction = model.predict(df)

    return f"🏠 Estimated House Price: ₹{round(prediction[0], 2)} Lakh"


# =========================
# Gradio UI
# =========================

with gr.Blocks(title="House Price Prediction") as app:

    gr.Markdown("# 🏠 House Price Prediction")

    with gr.Row():

        with gr.Column():

            area_type = gr.Dropdown(
                choices=area_types,
                label="Area Type",
                value=area_types[0] if area_types else None
            )

            availability = gr.Dropdown(
                choices=available,
                label="Availability",
                value=available[0] if available else None
            )

            location = gr.Dropdown(
                choices=locations,
                label="Location",
                value=locations[0] if locations else None
            )

            size_bhk = gr.Number(
                label="BHK Size",
                minimum=1,
                precision=0,
                value=2
            )

        with gr.Column():

            total_sqft = gr.Number(
                label="Total Size (sqft)",
                minimum=100,
                value=1000
            )

            bath = gr.Number(
                label="Number of Bathrooms",
                minimum=1,
                precision=0,
                value=2
            )

            balcony = gr.Number(
                label="Number of Balconies",
                minimum=0,
                precision=0,
                value=1
            )

    predict_button = gr.Button(
        "Predict House Price",
        variant="primary"
    )

    output = gr.Textbox(
        label="Prediction",
        interactive=False
    )

    predict_button.click(
        fn=predict_house_price,
        inputs=[
            area_type,
            availability,
            location,
            size_bhk,
            total_sqft,
            bath,
            balcony
        ],
        outputs=output
    )


# =========================
# Run App
# =========================

if __name__ == "__main__":
    app.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
